# Lesson 3: Advantage Actor-Critic / A2C on CartPole

A2C is the next step after vanilla policy gradient.

REINFORCE used full returns directly:

```text
make actions more likely if the episode return was high
```

A2C adds a critic that estimates how good the current state is. Then the actor learns from the **advantage**:

```text
advantage = actual return - critic's expected value
```

This answers a better question:

```text
Was this action better or worse than expected from this state?
```


## 1) Imports


In [ ]:
%pip install -U "gymnasium[classic-control]"

import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

print("gymnasium:", gym.__version__)
print("torch:", torch.__version__)


## 2) Seeds and Device


In [ ]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 3) Environments

Same CartPole setup as before, using modern Gymnasium.


In [ ]:
train_env = gym.make("CartPole-v1")
test_env = gym.make("CartPole-v1")

train_env.action_space.seed(SEED)
test_env.action_space.seed(SEED + 1)

state, info = train_env.reset(seed=SEED)
print("example state:", state)
print("observation space:", train_env.observation_space)
print("action space:", train_env.action_space)


## 4) Actor-Critic Network

The actor chooses actions. The critic predicts state value.

We use one shared hidden layer, then two heads:

```text
state -> shared features -> actor logits
                       -> critic value
```

The actor output has size 2 for left/right. The critic output is one scalar: `V(s)`.


In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
        self.actor = nn.Linear(hidden_dim, output_dim)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        features = self.shared(x)
        logits = self.actor(features)
        value = self.critic(features).squeeze(-1)
        return logits, value


## 5) Build Policy and Optimizer


In [ ]:
INPUT_DIM = train_env.observation_space.shape[0]
HIDDEN_DIM = 128
OUTPUT_DIM = train_env.action_space.n

policy = ActorCritic(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
optimizer = optim.Adam(policy.parameters(), lr=1e-2)

print(policy)


## 6) Select an Action

We sample actions from `Categorical(logits=logits)` for exploration. We also keep:

- `log_prob`: needed for actor loss
- `value`: critic's estimate `V(s)`
- `entropy`: optional exploration bonus


In [ ]:
def select_action(policy, state):
    state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    logits, value = policy(state_tensor)
    distribution = Categorical(logits=logits)
    action = distribution.sample()
    log_prob = distribution.log_prob(action)
    entropy = distribution.entropy()
    return action.item(), log_prob.squeeze(0), value.squeeze(0), entropy.squeeze(0)


## 7) Returns

A2C still needs returns as the target for the critic.

The critic tries to predict these returns.


In [ ]:
def calculate_returns(rewards, discount_factor, normalize=True):
    returns = []
    running_return = 0.0

    for reward in reversed(rewards):
        running_return = reward + discount_factor * running_return
        returns.insert(0, running_return)

    returns = torch.as_tensor(returns, dtype=torch.float32, device=device)

    if normalize and len(returns) > 1:
        std = returns.std(unbiased=False)
        if std > 1e-8:
            returns = (returns - returns.mean()) / (std + 1e-8)

    return returns


## 8) Advantages

The advantage tells the actor whether the outcome was better or worse than expected.

```text
advantage = return - value
```

If advantage is positive, increase the probability of the action. If negative, decrease it.


In [ ]:
def calculate_advantages(returns, values, normalize=True):
    values = torch.stack(values)
    advantages = returns - values.detach()

    if normalize and len(advantages) > 1:
        std = advantages.std(unbiased=False)
        if std > 1e-8:
            advantages = (advantages - advantages.mean()) / (std + 1e-8)

    return advantages, values


## 9) Update Actor and Critic

A2C has two losses:

```text
actor loss  = -(advantage * log_prob)
critic loss = MSE(value, return)
```

The actor learns what actions to prefer. The critic learns to predict returns.


In [ ]:
def update_policy(advantages, log_probs, returns, values, entropies, optimizer):
    log_probs = torch.stack(log_probs)
    entropies = torch.stack(entropies)

    policy_loss = -(advantages.detach() * log_probs).sum()
    value_loss = F.mse_loss(values, returns)
    entropy_bonus = entropies.mean()

    loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_bonus

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return policy_loss.item(), value_loss.item(), loss.item()


## 10) Train One Episode


In [ ]:
def train_one_episode(env, policy, optimizer, discount_factor, seed=None):
    policy.train()

    log_probs = []
    values = []
    rewards = []
    entropies = []
    episode_reward = 0.0

    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action, log_prob, value, entropy = select_action(policy, state)
        next_state, reward, terminated, truncated, info = env.step(action)

        log_probs.append(log_prob)
        values.append(value)
        rewards.append(reward)
        entropies.append(entropy)
        episode_reward += reward
        state = next_state

    returns = calculate_returns(rewards, discount_factor)
    advantages, values = calculate_advantages(returns, values)
    policy_loss, value_loss, total_loss = update_policy(
        advantages, log_probs, returns, values, entropies, optimizer
    )

    return policy_loss, value_loss, total_loss, episode_reward


## 11) Evaluate


In [ ]:
def evaluate(env, policy, seed=None):
    policy.eval()

    episode_reward = 0.0
    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False

    while not (terminated or truncated):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

        with torch.no_grad():
            logits, value = policy(state_tensor)
            action = torch.argmax(logits, dim=-1).item()

        state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

    return episode_reward


## 12) Training Loop


In [ ]:
MAX_EPISODES = 500
DISCOUNT_FACTOR = 0.99
N_TRIALS = 25
REWARD_THRESHOLD = 475
PRINT_EVERY = 10

train_rewards = []
test_rewards = []
policy_losses = []
value_losses = []
recent_test_rewards = deque(maxlen=N_TRIALS)

for episode in range(1, MAX_EPISODES + 1):
    policy_loss, value_loss, total_loss, train_reward = train_one_episode(
        train_env, policy, optimizer, DISCOUNT_FACTOR, seed=SEED + episode
    )
    test_reward = evaluate(test_env, policy, seed=SEED + 10_000 + episode)

    train_rewards.append(train_reward)
    test_rewards.append(test_reward)
    policy_losses.append(policy_loss)
    value_losses.append(value_loss)
    recent_test_rewards.append(test_reward)

    if episode % PRINT_EVERY == 0:
        print(
            f"| Episode: {episode:3} | "
            f"Mean Train: {np.mean(train_rewards[-N_TRIALS:]):6.1f} | "
            f"Mean Test: {np.mean(recent_test_rewards):6.1f} | "
            f"Policy Loss: {policy_loss:8.2f} | Value Loss: {value_loss:8.2f} |"
        )

    if len(recent_test_rewards) == N_TRIALS and np.mean(recent_test_rewards) >= REWARD_THRESHOLD:
        print(f"Reached reward threshold in {episode} episodes")
        break


## 13) Plot Rewards


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(train_rewards, label="Train Reward", alpha=0.7)
plt.plot(test_rewards, label="Test Reward")
plt.axhline(REWARD_THRESHOLD, color="red", linestyle="--", label="Threshold")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(True)
plt.show()


## 14) Watch Policy


In [ ]:
def watch_policy(policy, seed=SEED, max_steps=500):
    render_env = gym.make("CartPole-v1", render_mode="human")
    state, info = render_env.reset(seed=seed)
    terminated = False
    truncated = False
    total_reward = 0.0
    steps = 0

    while not (terminated or truncated) and steps < max_steps:
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits, value = policy(state_tensor)
            action = torch.argmax(logits, dim=-1).item()
        state, reward, terminated, truncated, info = render_env.step(action)
        total_reward += reward
        steps += 1

    print(f"Episode finished. Total reward: {total_reward}, steps: {steps}")
    render_env.close()


# Uncomment after training if your machine supports GUI rendering.
# watch_policy(policy)


## 15) Exercises

1. Print one batch of `returns`, `values`, and `advantages`. Which values are positive?
2. Set the entropy coefficient in `update_policy` to `0.0`. Does training change?
3. Explain why `advantages.detach()` is used in the actor loss.

## 16) Mental Model

A2C = actor + critic.

```text
critic: how good is this state normally?
actor: was my sampled action better or worse than that expectation?
```

The critic lowers variance compared with vanilla REINFORCE.
